# Audio Analyser → Visual Prompt → seed42 Control API

This notebook listens to music, describes it, and turns that description into a short visual prompt for the seed42 SDXL-Turbo stream.
The feature extractor, scene tracker and prompt writer match the server code in `AudioAnalyser_clean_v3` (`features.py`, `scene.py`, `prompt.py`, `jobs.run_window`), written here as plain notebook functions.

**Pipeline:** audio → features (tempo, energy phase, band, genre, mood, lyric themes) → input sentence → **scene decision** → LLM (only when the scene changes) → 40-word prompt → Control API

| Tool | What it provides |
|---|---|
| Librosa | Energy (RMS) and its phase, fallback tempo, frequency bands, silence gaps |
| Madmom | Neural beat tracking → BPM |
| Essentia | Genre (Discogs 400) and mood scores (happy, sad, relaxed, aggressive, party) |
| Faster-Whisper | Lyric keywords ("themes") |
| Scene tracker | Compares each window with the music the current scene was written for: **hold**, **evolve**, **blend**, **new scene**, **new track** or **revert**; time also moves a steady scene on (**drift** and **timed** changes) |
| Groq (`openai/gpt-oss-120b`) | Writes a new prompt, evolves it, or blends toward the new music |

### Before you run
1. **Runtime:** GPU recommended (Runtime → Change runtime type). CPU works but each window takes ~7–11 s.
2. **Audio:** upload your tracks to `/content` (Files panel) and list them in **Configuration**.
3. **LLM key:** create a key at console.groq.com, add it in 🔑 **Secrets** under the name set in `GROQ_SECRET_NAME`, and turn on notebook access. Without a key, everything except the LLM step still runs (the scene is then held).

Then use **Runtime → Run all**. Sections run top to bottom.

### Control API
The **Mock Server Connection** cells at the top start a local mock of the seed42 Control API and create a stream. The full pipeline (section 7.2) sends its prompts to that stream with `client.send_prompt()`.

### Baseline vs new pipeline
Section 8 compares the simplest end-to-end pipeline (Librosa only, template prompt, no scenes) with the new one on the same windows.

**Mock Server Connection**

In [1]:
%%writefile mock-control-api.py

#!/usr/bin/env python3
"""Mock of the seed42 Control API (/api/streams), for integration development.

Live streams are not a good build target: they cost real time to start, and an
unreliable one is indistinguishable from a bug in your own client. This
reproduces the contract in control-api.md closely enough to build the
whole request lifecycle against, on a laptop, for free.

    python mock-control-api.py
    # http://127.0.0.1:8787   token=test-token

Deliberately unfriendly where the real API is unfriendly, because a mock that
only serves the happy path teaches a client to crash in production:

  * a missing or wrong token is a real 401, not a warning
  * create is followed by a window of not-ready, so the retry loop is exercised
    rather than skipped
  * --not-ready 409 answers the other legal not-ready code, so a client that
    only handles 404 fails here rather than later
  * a PATCH carrying a creation-time field is answered 200, exactly as the real
    API would, and warns on stdout. That is the failure this file most exists
    to catch: it is invisible in the response, and in production it reloads the
    pipeline and stops the show for ~30 seconds.

Standard library only, on purpose: nobody should have to install anything to
test against this.
"""
import argparse
import json
import random
import secrets
import sys
import threading
import time
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.parse import urlparse, parse_qs

# Fields fixed when the stream is created. Sending one in a PATCH forces a
# pipeline reload on the real API. Mirrors control-api.md section 3.
COLD_FIELDS = {
    "model_id", "width", "height", "acceleration",
    "lora_dict", "use_lcm_lora", "filters",
}
COLD_CONTROLNET_FIELDS = {"model_id", "preprocessor", "preprocessor_params"}

STREAMS = {}
LOCK = threading.Lock()
CFG = None


def now():
    return time.monotonic()


class Stream:
    """One mock stream. ready_at is when it stops answering not-ready;
    dies_at is set only under --unstable."""

    def __init__(self, params):
        self.id = "str_" + secrets.token_hex(8)
        self.params = dict(params or {})
        self.created = now()
        self.ready_at = self.created + CFG.ready_after
        self.dies_at = self.created + 45 if CFG.unstable else None
        self.patches = 0

    @property
    def state(self):
        if self.dies_at and now() >= self.dies_at:
            return "dead"
        return "ready" if now() >= self.ready_at else "starting"


class Handler(BaseHTTPRequestHandler):
    protocol_version = "HTTP/1.1"

    # -- plumbing ---------------------------------------------------------
    def log_message(self, fmt, *args):
        print("  %s" % (fmt % args))

    def _send(self, code, payload):
        body = json.dumps(payload).encode()
        self.send_response(code)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.send_header("Access-Control-Allow-Origin", "*")
        self.send_header("Access-Control-Allow-Methods", "GET, POST, PATCH, DELETE, OPTIONS")
        self.send_header("Access-Control-Allow-Headers", "Content-Type, Authorization")
        self.end_headers()
        self.wfile.write(body)

    def _body(self):
        n = int(self.headers.get("Content-Length") or 0)
        if not n:
            return {}
        try:
            return json.loads(self.rfile.read(n) or b"{}")
        except json.JSONDecodeError:
            return None

    def _authed(self):
        """The real endpoint gates every verb but OPTIONS. Same here: a client
        that never sees a 401 in development meets its first one live."""
        header = self.headers.get("Authorization", "")
        if header.startswith("Bearer ") and header[7:] == CFG.token:
            return True
        self._send(401, {"error": "Not authenticated"})
        return False

    def _maybe_fail(self):
        if CFG.fail_rate and random.random() < CFG.fail_rate:
            self._send(502, {"error": "Upstream error",
                             "details": "injected by --fail-rate"})
            return True
        return False

    def _login(self):
        body = self._body()
        if body is None:
            return self._send(400, {"error": "Invalid JSON"})
        if body.get("action") != "login":
            return self._send(400, {"error": "Unknown action"})
        if not body.get("email") or not body.get("password"):
            return self._send(400, {"error": "email and password required"})
        print("  login %s -> token" % body["email"])
        return self._send(200, {"ok": True, "token": CFG.token})

    def _not_ready(self):
        """The one retryable case. Both codes are legal; a client must treat
        them alike, so the mock can be told to answer either."""
        if CFG.not_ready == 409:
            return self._send(409, {"error": "Stream is not running", "status": "starting"})
        return self._send(404, {"error": "stream not ready"})

    # -- verbs ------------------------------------------------------------
    def do_OPTIONS(self):
        self._send(200, {})

    def do_GET(self):
        if not self._authed():
            return
        # Only meaningful after a 202. A client that polls unconditionally
        # should discover that here rather than in production.
        if not CFG.async_create:
            return self._send(404, {"error": "No status endpoint for this stream"})
        sid = (parse_qs(urlparse(self.path).query).get("id") or [None])[0]
        if not sid:
            return self._send(400, {"error": "id is required"})
        with LOCK:
            s = STREAMS.get(sid)
        if not s:
            return self._send(404, {"error": "Stream not found"})
        if s.state == "dead":
            return self._send(502, {"id": sid, "status": "failed",
                                    "error": "stream ended before it was ready"})
        if s.state == "ready":
            return self._send(200, {"id": sid, "status": "ready"})
        return self._send(200, {"id": sid, "status": "starting"})

    def do_POST(self):
        # /api/auth is the one call that carries no token, because it is how
        # you get one. Routed by path so a client written against the real
        # API -- login first, then streams -- runs here unchanged. Any
        # non-empty email and password are accepted; the answer is the
        # mock's own token.
        if urlparse(self.path).path.rstrip("/").endswith("/api/auth"):
            return self._login()
        if not self._authed():
            return
        if self._maybe_fail():
            return
        body = self._body()
        if body is None:
            return self._send(400, {"error": "Invalid JSON"})
        s = Stream(body.get("params"))
        with LOCK:
            STREAMS[s.id] = s
        print("  created %s (ready in %.0fs%s)"
              % (s.id, CFG.ready_after, ", ends at 45s" if s.dies_at else ""))
        if CFG.async_create:
            return self._send(202, {"id": s.id, "status": "starting"})
        return self._send(201, {
            "id": s.id,
            "stream_key": "stk_" + secrets.token_hex(8),
            "output_stream_url": "rtmp://mock.local/live/" + secrets.token_hex(4),
            "params": s.params,
        })

    def do_PATCH(self):
        if not self._authed():
            return
        if self._maybe_fail():
            return
        body = self._body()
        if body is None:
            return self._send(400, {"error": "Invalid JSON"})
        sid = body.get("id")
        if not sid:
            return self._send(400, {"error": "Stream id is required for PATCH"})
        with LOCK:
            s = STREAMS.get(sid)
        if not s:
            return self._send(404, {"error": "Stream not found"})
        if s.state == "dead":
            return self._send(502, {"error": "stream ended"})
        if s.state == "starting":
            return self._not_ready()

        params = body.get("params") or {}
        self._warn_cold(params)
        s.patches += 1
        s.params.update(params)
        print("  patch #%d on %s: %s" % (s.patches, sid, _summarise(params)))
        return self._send(200, {"ok": True, "id": sid})

    def do_DELETE(self):
        if not self._authed():
            return
        body = self._body()
        if body is None:
            return self._send(400, {"error": "Invalid JSON"})
        sid = (body or {}).get("stream_id") or (body or {}).get("id")
        if not sid:
            return self._send(400, {"error": "Stream id is required for DELETE"})
        with LOCK:
            existed = STREAMS.pop(sid, None) is not None
        if CFG.unstable:
            # DELETE is allowed to fail. Clients must tolerate it, not retry it.
            return self._send(502, {"error": "Upstream error"})
        print("  deleted %s%s" % (sid, "" if existed else " (unknown)"))
        return self._send(200, {"ok": True, "id": sid})

    # -- the check the real API will not do for you -----------------------
    def _warn_cold(self, params):
        hits = sorted(COLD_FIELDS.intersection(params))
        for cn in params.get("controlnets") or []:
            if isinstance(cn, dict):
                hits += ["controlnets[].%s" % f
                         for f in sorted(COLD_CONTROLNET_FIELDS.intersection(cn))]
        if hits:
            print("\n  * COLD FIELD IN PATCH: %s" % ", ".join(hits))
            print("      Production would reload the pipeline (~30s) and the")
            print("      performance would stop. Send only hot fields.")
            print("      See control-api.md section 3.\n")


def _summarise(params):
    bits = []
    if "prompt" in params:
        p = str(params["prompt"])
        bits.append("prompt=%r" % (p[:40] + "..." if len(p) > 40 else p))
    for k in ("t_index_list", "guidance_scale", "delta", "seed", "num_inference_steps"):
        if k in params:
            bits.append("%s=%s" % (k, params[k]))
    cns = params.get("controlnets")
    if isinstance(cns, list):
        bits.append("cn=[%s]" % ",".join(
            str(c.get("conditioning_scale")) for c in cns if isinstance(c, dict)))
    return " ".join(bits) or "(empty)"


def main():
    global CFG
    ap = argparse.ArgumentParser(description=__doc__.split("\n")[0])
    ap.add_argument("--port", type=int, default=8787)
    ap.add_argument("--token", default="test-token")
    ap.add_argument("--not-ready", type=int, choices=(404, 409), default=404,
                    dest="not_ready",
                    help="which not-ready code to answer (both are legal; your "
                         "client must handle either)")
    ap.add_argument("--async-create", action="store_true", dest="async_create",
                    help="answer 202 on create and require the GET poll")
    ap.add_argument("--ready-after", type=float, default=4.0,
                    help="seconds a new stream answers not-ready (default 4)")
    ap.add_argument("--always-ready", action="store_true",
                    help="skip the not-ready window entirely")
    ap.add_argument("--unstable", action="store_true",
                    help="streams end ~45s after creation and DELETE fails")
    ap.add_argument("--fail-rate", type=float, default=0.0,
                    help="probability of a 502 on create/patch, e.g. 0.2")
    CFG = ap.parse_args()
    if CFG.always_ready:
        CFG.ready_after = 0.0

    # Line-buffer stdout. Python block-buffers when it is not a terminal, so
    # piping this to a file or running it from an IDE swallowed every warning
    # until the process exited -- including the cold-field warning, which is
    # the main reason to run this at all.
    try:
        sys.stdout.reconfigure(line_buffering=True)
    except AttributeError:  # pragma: no cover - Python < 3.7
        pass

    srv = ThreadingHTTPServer(("127.0.0.1", CFG.port), Handler)
    print("seed42 Control API mock")
    print("  http://127.0.0.1:%d   token=%s" % (CFG.port, CFG.token))
    print("  Authorization: Bearer %s" % CFG.token)
    print("  not-ready=%d%s%s"
          % (CFG.not_ready,
             "  async-create" if CFG.async_create else "",
             "  unstable" if CFG.unstable else ""))
    print("  contract: control-api.md      ctrl-c to stop\n")
    try:
        srv.serve_forever()
    except KeyboardInterrupt:
        print("\nstopped")


if __name__ == "__main__":
    main()

Writing mock-control-api.py


In [2]:
import subprocess, time
log_file = open('server.log', 'w')
proc = subprocess.Popen(['python3', 'mock-control-api.py'],
                         stdout=log_file, stderr=subprocess.STDOUT,
                         start_new_session=True)
time.sleep(2)
print("Started. Exit code:", proc.poll())

Started. Exit code: None


In [3]:
proc = subprocess.Popen(['python3', 'mock-control-api.py', '--not-ready', '409'],
                         stdout=open('server.log', 'w'), stderr=subprocess.STDOUT,
                         start_new_session=True)
time.sleep(2)
print("Exit code:", proc.poll())

Exit code: 1


In [4]:
!curl -s http://127.0.0.1:8787/api/streams

{"error": "Not authenticated"}

In [5]:
import time
import requests

class Seed42Client:
    def __init__(self, base_url, timeout_seconds=12):
        self.base_url = base_url.rstrip("/")
        self.timeout = timeout_seconds
        self.token = None

    def login(self, email, password):
        r = requests.post(f"{self.base_url}/api/auth",
            json={"action": "login", "email": email, "password": password},
            timeout=self.timeout)
        r.raise_for_status()
        self.token = r.json()["token"]
        return self.token

    def _headers(self):
        return {"Authorization": f"Bearer {self.token}"}

    def create_stream(self, initial_prompt="abstract shapes, vivid, neon"):
        r = requests.post(f"{self.base_url}/api/streams", headers=self._headers(),
            json={"params": {"model_id": "stabilityai/sdxl-turbo",
                              "width": 1024, "height": 576, "prompt": initial_prompt}},
            timeout=self.timeout)
        r.raise_for_status()
        data = r.json()
        return data["id"], r.status_code == 202

    def get_status(self, stream_id):
        r = requests.get(f"{self.base_url}/api/streams", headers=self._headers(),
            params={"id": stream_id}, timeout=self.timeout)
        if r.status_code == 404:
            return "not_found"
        r.raise_for_status()
        return r.json().get("status")

    def wait_until_ready(self, stream_id, poll_every=2, max_wait=30):
        waited = 0
        while waited < max_wait:
            if self.get_status(stream_id) == "ready":
                return True
            time.sleep(poll_every)
            waited += poll_every
        return False

    def send_prompt(self, stream_id, prompt, max_retries=3, retry_wait=2):
        attempt = 0
        while True:
            attempt += 1
            r = requests.patch(f"{self.base_url}/api/streams", headers=self._headers(),
                json={"id": stream_id, "params": {"prompt": prompt}},
                timeout=self.timeout)
            if r.status_code in (404, 409) and attempt <= max_retries:
                time.sleep(retry_wait)
                continue
            r.raise_for_status()
            return r.json()

    def end_stream(self, stream_id):
        try:
            r = requests.delete(f"{self.base_url}/api/streams", headers=self._headers(),
                json={"stream_id": stream_id}, timeout=self.timeout)
            return r.status_code == 200
        except requests.RequestException:
            return False

print("Seed42Client defined.")

Seed42Client defined.


In [6]:
client = Seed42Client("http://127.0.0.1:8787")
client.login("test@example.com", "anything")

stream_id, needs_poll = client.create_stream()
if needs_poll:
    client.wait_until_ready(stream_id)

print("Stream ready:", stream_id)

Stream ready: str_b22275f29e549e6a


In [7]:
!pkill -f mock-control-api.py
time.sleep(2)

---
## 1. Setup
### 1.1 Install packages

In [8]:
!apt-get -qq update && apt-get -qq install -y ffmpeg
!pip install -q librosa soundfile essentia-tensorflow cython git+https://github.com/CPJKU/madmom.git faster-whisper torch openai requests

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 291.5/291.5 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 MB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 73.2 MB/s eta 0:00:00


### 1.2 Configuration
Everything you may need to change lives in this one cell.

In [9]:
# --- Audio -------------------------------------------------------------------
# Upload tracks to /content and list them here. Missing files are skipped.
AUDIO_PLAYLIST = [
    "/content/036-1896735-Revolution Void-Effects of Elevation _Citizen Prime Remix_.mp3",
    "/content/046-1875274-Axined _amp_ Mr. Moohman-How Do I Get My Grass Really Green.mp3",
    "/content/003-2235411-Jackson Frost-Fashionable Progressive House.mp3",
]
SR = 22050                 # every track is resampled to this rate for analysis
N_THEMES = 4               # lyric keywords kept per window

# --- Playlist engine ---------------------------------------------------------
WINDOW_SEC = 10            # audio analysed per update
STEP_SEC = 5               # one update (scene decision) every STEP_SEC seconds

# --- LLM -------------------------------------------------------------------------
GROQ_SECRET_NAME = "seedAudio"            # name of the Colab secret holding your Groq key
LLM_MODEL        = "openai/gpt-oss-120b"
MAX_PROMPT_WORDS = 40                     # SDXL-Turbo reads ~77 tokens; 40 words stays under that

# --- Scenes ----------------------------------------------------------------------
# Each window gets a change score from 0 (same music) to 1 (different in every way).
SCENE_EVOLVE_AT   = 0.2    # at or above: adjust the scene
SCENE_CHANGE_AT   = 0.4    # at or above: blend, then replace if it lasts
SCENE_GAP_SEC     = 1.5    # near silence this long suggests a track change
SCENE_SILENCE_RMS = 0.01   # window energy under this counts as silence
SCENE_MAX_SEC     = 60     # a scene this old is replaced even if the music holds (0 = never)
SCENE_DRIFT_SEC   = 15     # evolve at least this often within a scene (0 = only when the music changes)

# --- Baseline vs new comparison (section 8) ------------------------------------
LABELS_CSV = "/content/labels.csv"   # optional ground truth: file,genre,bpm,boundaries
RUN_JUDGE  = False                   # True: the LLM scores every prompt 1-5 (2 calls per window)

print("Configuration set.")

Configuration set.


### 1.3 Load models
Imports, compatibility shims for Madmom on newer NumPy, then the Essentia (embeddings, genre, mood) and Whisper models. Model files download once and are reused.

Expected output ends with `Mood models loaded: [...]` listing all five moods and `Whisper (tiny): LOADED`.

In [10]:
# =============================================================================
# 1. IMPORTS
# =============================================================================
import os
import io
import time
import json
import logging
import warnings
import urllib.request
import collections
import collections.abc
from collections import Counter

import numpy as np
import soundfile as sf
import librosa
import torch

# Harmless noise: NumPy alias warnings from the shims below, and the
# Hugging Face "unauthenticated request" notice when Whisper downloads.
warnings.filterwarnings("ignore", category=FutureWarning)
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)

# =============================================================================
# 2. NUMPY / COLLECTIONS SHIMS (must run before importing madmom)
# =============================================================================
for _name, _py in {"float": float, "int": int, "complex": complex,
                   "object": object, "bool": bool, "str": str}.items():
    if not hasattr(np, _name):
        setattr(np, _name, _py)

for _name, _val in {"NINF": -np.inf, "PINF": np.inf, "Inf": np.inf,
                    "float_": np.float64, "complex_": np.complex128}.items():
    if not hasattr(np, _name):
        setattr(np, _name, _val)

for _abc in ("MutableSequence", "Iterable", "Callable", "MutableMapping", "Mapping", "Sequence"):
    if not hasattr(collections, _abc):
        setattr(collections, _abc, getattr(collections.abc, _abc))

# =============================================================================
# 3. MODEL LIBRARIES
# =============================================================================
MADMOM_AVAILABLE = False
try:
    from madmom.features.beats import RNNBeatProcessor, DBNBeatTrackingProcessor
    MADMOM_AVAILABLE = True
    print("Madmom neural beat processor: LOADED")
except Exception as e:
    print("Madmom unavailable, falling back to Librosa tempo:", e)

from essentia.standard import TensorflowPredictEffnetDiscogs, TensorflowPredict2D
from faster_whisper import WhisperModel

# =============================================================================
# 4. HARDWARE
# =============================================================================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
COMPUTE_TYPE = "float16" if DEVICE == "cuda" else "int8"
print(f"Running pipeline on: {DEVICE.upper()} (compute mode: {COMPUTE_TYPE})")

# =============================================================================
# 5. DOWNLOAD ESSENTIA MODELS (embeddings, genre, mood)
# =============================================================================
MODEL_DIR = "essentia_models"
MODEL_BASE = "https://essentia.upf.edu/models"
ESSENTIA_SR = 16000
MOOD_NAMES = ["happy", "sad", "relaxed", "aggressive", "party"]

os.makedirs(MODEL_DIR, exist_ok=True)

MODEL_FILES = {
    "embeddings": ("feature-extractors/discogs-effnet", "discogs-effnet-bs64-1.pb"),
    "genre":      ("classification-heads/genre_discogs400", "genre_discogs400-discogs-effnet-1.pb"),
    "genre_meta": ("classification-heads/genre_discogs400", "genre_discogs400-discogs-effnet-1.json"),
}
for m in MOOD_NAMES:
    for ext in (".pb", ".json"):
        MODEL_FILES[f"mood_{m}{ext}"] = (f"classification-heads/mood_{m}",
                                         f"mood_{m}-discogs-effnet-1{ext}")

for subdir, fname in MODEL_FILES.values():
    dest = os.path.join(MODEL_DIR, fname)
    if not os.path.exists(dest):
        urllib.request.urlretrieve(f"{MODEL_BASE}/{subdir}/{fname}", dest)

# =============================================================================
# 6. EMBEDDING + GENRE MODELS
# =============================================================================
embedding_model = None
genre_head = None
genre_classes = []
ESSENTIA_MODELS_LOADED = False

try:
    embedding_model = TensorflowPredictEffnetDiscogs(
        graphFilename=os.path.join(MODEL_DIR, "discogs-effnet-bs64-1.pb"),
        output="PartitionedCall:1",
    )
    with open(os.path.join(MODEL_DIR, "genre_discogs400-discogs-effnet-1.json")) as f:
        genre_classes = json.load(f)["classes"]

    genre_head = TensorflowPredict2D(
        graphFilename=os.path.join(MODEL_DIR, "genre_discogs400-discogs-effnet-1.pb"),
        input="serving_default_model_Placeholder",
        output="PartitionedCall",
    )
    ESSENTIA_MODELS_LOADED = True
    print("Essentia embedding + genre models: LOADED")
except Exception as e:
    print(f"ERROR loading Essentia models: {e}")
    print("Genre and mood will be reported as 'unavailable'.")

# =============================================================================
# 7. MOOD MODELS (separate, so a mood failure never disables genre)
# =============================================================================
mood_heads = {}
for m in MOOD_NAMES:
    try:
        with open(os.path.join(MODEL_DIR, f"mood_{m}-discogs-effnet-1.json")) as f:
            meta = json.load(f)
        # Input/output tensor names differ between Essentia model releases,
        # so read them from the model's own metadata.
        schema = meta.get("schema", {})
        inp = schema.get("inputs", [{}])[0].get("name", "model/Placeholder")
        out = next((o["name"] for o in schema.get("outputs", [])
                    if o.get("output_purpose") == "predictions"), "model/Softmax")
        head = TensorflowPredict2D(
            graphFilename=os.path.join(MODEL_DIR, f"mood_{m}-discogs-effnet-1.pb"),
            input=inp, output=out,
        )
        mood_heads[m] = (head, meta["classes"].index(m))   # index of e.g. "happy" vs "non_happy"
    except Exception as e:
        print(f"Mood model '{m}' skipped: {e}")
print("Mood models loaded:", list(mood_heads))

# =============================================================================
# 8. WHISPER + LYRIC STOPWORDS
# =============================================================================
whisper_model = WhisperModel("tiny", device=DEVICE, compute_type=COMPUTE_TYPE)
STOPWORDS = set("a an the and or but if then than that this in on into to for with "
                "without about oh yeah uh ah hey".split())
print("Whisper (tiny): LOADED")

Madmom neural beat processor: LOADED
Running pipeline on: CUDA (compute mode: float16)
Essentia embedding + genre models: LOADED
Mood models loaded: ['happy', 'sad', 'relaxed', 'aggressive', 'party']


Whisper (tiny): LOADED


---
## 2. Feature extraction

`analyse_window()` analyses one window of audio entirely in memory (no temp files):

1. **Librosa:** RMS energy (how loud and dense the window is) and a fallback tempo
2. **Madmom:** detects individual beats, then BPM = 60 ÷ median gap between beats
3. **Essentia:** resamples to 16 kHz, predicts genre (top 2 with scores) and mood (up to 2 moods above 50%, otherwise `neutral`), and keeps **every mood score** so the scene tracker can measure how much the mood moved
4. **Whisper:** transcribes any vocals and keeps the most frequent meaningful words as "themes" (or `instrumental`)
5. **Energy phase and band:** a short key next to each readable label, used by the scene tracker and the baseline

| RMS | Phase key | Label |
|---|---|---|
| < 0.15 | `breakdown` | Low Energy / Breakdown Phase |
| 0.15–0.40 | `groove` | Moderate Energy / Steady Groove Phase |
| 0.40–0.65 | `build` | High Energy / Build-up Phase |
| ≥ 0.65 | `drop` | Peak Energy / Drop Phase |

Bands: `sub` (20–120 Hz, kick and bass), `mid` (120 Hz–2 kHz, vocals and synths), `high` (2–8 kHz, hi-hats and percussion).

`build_llm_input()` combines everything into one sentence for the LLM. If a model failed to load, its part is left out instead of sending the word "unavailable". Each window also reports per-stage timings in ms.

In [11]:
# Loudness thresholds for the four energy phases
PHASES = [  # (upper RMS bound, key, label)
    (0.15, "breakdown", "Low Energy / Breakdown Phase"),
    (0.40, "groove", "Moderate Energy / Steady Groove Phase"),
    (0.65, "build", "High Energy / Build-up Phase"),
    (float("inf"), "drop", "Peak Energy / Drop Phase"),
]

# Frequency bands; the loudest one describes the track's focus
BANDS = [  # (key, label, lo Hz, hi Hz)
    ("sub", "Low-End / Sub-Bass Focus (20 Hz - 120 Hz)", 20, 120),
    ("mid", "Mid-Range / Vocal & Synth Focus (120 Hz - 2 kHz)", 120, 2000),
    ("high", "High-End / Percussion & Hi-Hat Focus (2 kHz - 8 kHz)", 2000, 8000),
]


def get_mood(embeddings, threshold=0.5, top_k=2):
    """Up to `top_k` moods above `threshold` confidence, strongest first, or 'neutral';
    plus every score. One yes/no head per mood."""
    scores = {}
    for m, (head, idx) in mood_heads.items():
        try:
            scores[m] = float(np.mean(head(embeddings), axis=0)[idx])
        except Exception:
            pass
    picked = sorted(((s, m) for m, s in scores.items() if s >= threshold), reverse=True)[:top_k]
    return (", ".join(m for _, m in picked) if picked else "neutral"), scores


def extract_fast_features(y_slice, sr=SR, n_themes=3):
    """Runs Librosa, Madmom, Essentia and Whisper on one audio slice, entirely in RAM."""
    timings = {}

    # --- 1. Librosa: energy and fallback tempo ---------------------------------
    t = time.perf_counter()
    rms = float(np.sqrt(np.mean(y_slice ** 2)))
    librosa_bpm = round(float(np.atleast_1d(
        librosa.feature.tempo(y=y_slice, sr=sr, start_bpm=120))[0]), 1)
    timings["librosa"] = time.perf_counter() - t

    # --- 2. Madmom: neural beat tracking --------------------------------------
    madmom_bpm, beat_count = 0.0, 0
    if MADMOM_AVAILABLE:
        t = time.perf_counter()
        try:
            wav_io = io.BytesIO()                        # in-memory WAV, no disk writes
            sf.write(wav_io, y_slice, sr, format="WAV")
            wav_io.seek(0)
            beat_times = np.asarray(
                DBNBeatTrackingProcessor(fps=100)(RNNBeatProcessor()(wav_io)), dtype=float)
            if len(beat_times) >= 2:
                madmom_bpm = round(60.0 / float(np.median(np.diff(beat_times))), 1)
                beat_count = len(beat_times)
        except Exception:
            madmom_bpm = 0.0
        timings["madmom"] = time.perf_counter() - t

    if madmom_bpm > 0:
        bpm, bpm_source = madmom_bpm, "Madmom neural"
    else:
        bpm, bpm_source = librosa_bpm, "Librosa DSP"

    # --- 3. Essentia: genre and mood ------------------------------------------
    y16 = librosa.resample(y_slice, orig_sr=sr, target_sr=ESSENTIA_SR) if sr != ESSENTIA_SR else y_slice
    y16 = np.ascontiguousarray(y16, dtype=np.float32)

    mood_scores = {}
    if ESSENTIA_MODELS_LOADED:
        t = time.perf_counter()
        embeddings = embedding_model(y16)
        genre_probs = np.mean(genre_head(embeddings), axis=0)
        top_idx = np.argsort(genre_probs)[::-1][:2]
        top_genres = [genre_classes[i] for i in top_idx]
        genre_scores = [round(float(genre_probs[i]), 3) for i in top_idx]
        mood_str, mood_scores = get_mood(embeddings) if mood_heads else ("unavailable", {})
        timings["essentia"] = time.perf_counter() - t
    else:
        top_genres, genre_scores, mood_str = ["unavailable"], [], "unavailable"

    primary_genre = top_genres[0]
    secondary_genre = top_genres[1] if len(top_genres) > 1 else "N/A"
    genres_str = ", ".join(top_genres)

    # --- 4. Whisper: lyric keywords -------------------------------------------
    # Only word counts are kept; the transcript itself is never stored.
    if whisper_model is not None:
        t = time.perf_counter()
        segments, _ = whisper_model.transcribe(y16, beam_size=1, vad_filter=True)
        tokens = []
        for seg in segments:
            if getattr(seg, "no_speech_prob", 0.0) <= 0.6:
                words = "".join(c.lower() if c.isalpha() or c.isspace() else " " for c in seg.text).split()
                tokens.extend(w for w in words if len(w) >= 4 and w not in STOPWORDS)
        top_themes = [w for w, _ in Counter(tokens).most_common(n_themes)]
        themes_str = ", ".join(top_themes) if top_themes else "instrumental"
        timings["whisper"] = time.perf_counter() - t
    else:
        themes_str = "unavailable"

    return {
        "bpm": bpm,
        "bpm_source": bpm_source,
        "madmom_bpm": madmom_bpm,
        "librosa_bpm": librosa_bpm,
        "beat_count": beat_count,
        "energy": round(rms, 4),
        "top_2_genres": genres_str,
        "genre_scores": genre_scores,
        "primary_genre": primary_genre,
        "secondary_genre": secondary_genre,
        "mood": mood_str,
        "mood_scores": {k: round(v, 3) for k, v in mood_scores.items()},
        "lyric_themes": themes_str,
        "timings": {k: round(v * 1000) for k, v in timings.items()},
    }


def get_dominant_frequency_band(y_slice, sr):
    """(key, label) of the loudest of the three bands."""
    stft = np.abs(librosa.stft(y_slice, n_fft=2048, hop_length=512))
    freqs = librosa.fft_frequencies(sr=sr, n_fft=2048)

    def band_energy(lo, hi):
        mask = (freqs > lo) & (freqs <= hi)
        return float(np.mean(stft[mask, :])) if np.any(mask) else 0.0

    energies = {key: band_energy(lo, hi) for key, _, lo, hi in BANDS}
    key = max(energies, key=energies.get)
    label = next(lbl for k, lbl, _, _ in BANDS if k == key)
    return key, label


def get_energy_phase(rms):
    """(key, label) of the energy phase."""
    for bound, key, label in PHASES:
        if rms < bound:
            return key, label


def get_energy_descriptor(rms):
    return f"{rms:.3f} ({get_energy_phase(rms)[1]})"


def build_llm_input(features, energy_desc, dominant_band):
    """One sentence describing the window, used as the LLM's input.
    A missing model's part is left out instead of sending "unavailable"."""
    energy_phase = energy_desc.split("(")[1].rstrip(")")
    band_name = dominant_band.split("(")[0].strip()
    mood, genre, themes = features["mood"], features["primary_genre"], features["lyric_themes"]

    if "unavailable" not in (mood, genre, themes):
        genre_name = genre.split("---")[-1]    # "Electronic---House" -> "House"
        return (f"A {mood} mood, {energy_phase} {features['bpm']} BPM {genre_name} atmosphere, "
                f"featuring {band_name} with {themes} themes.")

    parts = [f"A {mood} mood," if mood != "unavailable" else "A",
             f"{energy_phase} {features['bpm']} BPM"]
    if genre != "unavailable":
        parts.append(genre.split("---")[-1])
    parts.append(f"atmosphere, featuring {band_name}")
    sentence = " ".join(parts)
    if themes != "unavailable":
        sentence += f" with {themes} themes"
    return sentence + "."


def analyse_window(y_slice, sr=SR):
    """Every feature for one window plus the LLM input sentence (no LLM, no API)."""
    t0 = time.perf_counter()
    features = extract_fast_features(y_slice, sr=sr, n_themes=N_THEMES)
    phase_key, phase_label = get_energy_phase(features["energy"])
    t = time.perf_counter()
    band_key, band_label = get_dominant_frequency_band(y_slice, sr)
    features["timings"]["bands"] = round((time.perf_counter() - t) * 1000)
    energy_desc = get_energy_descriptor(features["energy"])
    features.update({
        "phase": phase_key,
        "phase_label": phase_label,
        "band": band_key,
        "band_label": band_label,
        "llm_input": build_llm_input(features, energy_desc, band_label),
        "analysis_ms": round((time.perf_counter() - t0) * 1000),
    })
    return features


def window_starts(n_samples, window, step):
    """Window start offsets. A track shorter than one window is analysed whole."""
    if n_samples <= window:
        return [0]
    return list(range(0, n_samples - window + 1, step))

print("Feature extractor defined.")

Feature extractor defined.


---
## 3. Test on one track (sliding window)

Runs `analyse_window()` over the first track in `AUDIO_PLAYLIST`, without the LLM or the stream:
- **Window:** 20 seconds of audio analysed at a time
- **Step:** the window moves forward 5 seconds each time

This simulates a live stream on a pre-recorded file. If the file isn't found, a synthetic test tone is generated instead.

> On CPU each window takes ~7–11 s. Lower `window_sec` or raise `step_sec` for a faster run.

In [12]:
def run_one_track(audio_path, window_sec=20, step_sec=5):
    """Slides a window over one track and returns a list of feature dictionaries."""
    if not os.path.exists(audio_path):
        print(f"File '{audio_path}' not found. Using a synthetic test tone instead.")
        audio_path = "synthetic_party_track.wav"
        t = np.linspace(0, 120, 120 * SR, endpoint=False)
        synth = 0.5 * np.sin(2 * np.pi * 440 * t) * (np.sin(2 * np.pi * 2 * t) > 0)
        sf.write(audio_path, synth.astype(np.float32), SR)

    y, sr = librosa.load(audio_path, sr=SR, mono=True)
    window, step = int(window_sec * sr), int(step_sec * sr)

    print("=" * 110)
    print(" CONTINUOUS STREAM RUNNER (Librosa + Madmom + Essentia + Whisper)")
    print(f" Window: {window_sec}s | Step: every {step_sec}s | File: {os.path.basename(audio_path)}")
    print("=" * 110)
    print(f"{'Time':>8} | {'BPM':>6} | {'Energy':>7} | {'Phase':<9} | {'Top 2 genres':<34} | {'Mood':<18} | Proc")
    print("-" * 110)

    rows, t_run = [], time.time()
    for start in window_starts(len(y), window, step):
        end = min(start + window, len(y))
        rec = analyse_window(y[start:end], sr=sr)
        rec["t_end"] = round(end / sr, 1)
        rows.append(rec)
        print(f"{rec['t_end']:>7.1f}s | {rec['bpm']:>6.1f} | {rec['energy']:>7.4f} | {rec['phase']:<9} | "
              f"{rec['top_2_genres'][:34]:<34} | {rec['mood'][:18]:<18} | {rec['analysis_ms']:>5}ms")

    print("-" * 110)
    print(f"Done: {len(rows)} windows in {time.time() - t_run:.1f} s")
    return rows


output_stream = run_one_track(AUDIO_PLAYLIST[0], window_sec=20, step_sec=5)

 CONTINUOUS STREAM RUNNER (Librosa + Madmom + Essentia + Whisper)
 Window: 20s | Step: every 5s | File: 036-1896735-Revolution Void-Effects of Elevation _Citizen Prime Remix_.mp3
    Time |    BPM |  Energy | Phase     | Top 2 genres                       | Mood               | Proc
--------------------------------------------------------------------------------------------------------------
   20.0s |  133.3 |  0.1171 | breakdown | Electronic---Progressive Breaks, E | party, relaxed     |  8822ms
   25.0s |  133.3 |  0.1198 | breakdown | Electronic---Progressive Breaks, E | party, relaxed     |  6462ms
   30.0s |  133.3 |  0.1257 | breakdown | Electronic---Progressive Breaks, E | party, relaxed     |  6525ms
   35.0s |  130.4 |  0.1298 | breakdown | Electronic---Progressive Breaks, E | party, relaxed     |  6186ms
   40.0s |  133.3 |  0.1327 | breakdown | Electronic---Techno, Electronic--- | party, relaxed     |  6191ms
   45.0s |  130.4 |  0.1469 | breakdown | Electronic---Progressiv

### Inspect the results
The first window in full. `mood_scores` feeds the scene tracker's mood comparison, `timings` shows where the time goes, and `llm_input` is the sentence the LLM receives.

In [13]:
class NpEncoder(json.JSONEncoder):
    """Makes NumPy numbers and arrays JSON-serialisable."""
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super().default(obj)

if output_stream:
    print(json.dumps(output_stream[0], indent=2, cls=NpEncoder))

{
  "bpm": 133.3,
  "bpm_source": "Madmom neural",
  "madmom_bpm": 133.3,
  "librosa_bpm": 89.1,
  "beat_count": 44,
  "energy": 0.1171,
  "top_2_genres": "Electronic---Progressive Breaks, Electronic---Progressive House",
  "genre_scores": [
    0.489,
    0.274
  ],
  "primary_genre": "Electronic---Progressive Breaks",
  "secondary_genre": "Electronic---Progressive House",
  "mood": "party, relaxed",
  "mood_scores": {
    "happy": 0.181,
    "sad": 0.106,
    "relaxed": 0.556,
    "aggressive": 0.239,
    "party": 0.857
  },
  "lyric_themes": "instrumental",
  "timings": {
    "librosa": 1739,
    "madmom": 3959,
    "essentia": 2577,
    "whisper": 516,
    "bands": 23
  },
  "phase": "breakdown",
  "phase_label": "Low Energy / Breakdown Phase",
  "band": "sub",
  "band_label": "Low-End / Sub-Bass Focus (20 Hz - 120 Hz)",
  "llm_input": "A party, relaxed mood, Low Energy / Breakdown Phase 133.3 BPM Progressive Breaks atmosphere, featuring Low-End / Sub-Bass Focus with instrumental t

---
## 4. Prompt writer (Groq)

The LLM is `openai/gpt-oss-120b`, served by Groq through the OpenAI Python SDK. The key is read from the Colab secret named in `GROQ_SECRET_NAME`. Without it every call returns `FALLBACK_PROMPT`, and the scene tracker holds the current scene instead of jumping to it.

The scene tracker (section 5) calls one of three writers, only when the scene needs a new or revised prompt:

| Writer | Used for | Temperature |
|---|---|---|
| `generate_visual_prompt()` | first scene, new scene, new track: fresh subjects from a theme not used recently | 0.8 |
| `evolve_visual_prompt()` | moderate change or drift: keeps the scene, changes at most three phrases | 0.4 |
| `blend_visual_prompt()` | large change or silence gap: halfway between the current scene and the new audio | 0.6 |

**Why prompts are limited to 40 words:** SDXL-Turbo reads only about **77 tokens**. The limit is enforced three ways:
1. **Instructions:** one line of 8–12 short comma-separated phrases: one to three imaginative subjects (creatures, plants, surreal objects, landscapes) → setting → materials → lighting → atmosphere → camera. No sentences, numbers, genre names or text; no people, clubs, stages or speakers.
2. **Token cap:** `max_tokens=1000` with `reasoning_effort="low"` (gpt-oss counts its hidden reasoning in this budget).
3. **Python trim:** formatting is cleaned and anything over the limit is cut back to the last complete phrase.

In [14]:
from openai import OpenAI

GROQ_ENABLED, groq_client = False, None
try:
    from google.colab import userdata
    api_key = userdata.get(GROQ_SECRET_NAME)
    if not api_key:
        raise ValueError(f"secret '{GROQ_SECRET_NAME}' is empty")
    groq_client = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=str(api_key))
    GROQ_ENABLED = True
    print(f"Groq client ready (model: {LLM_MODEL}).")
except Exception as e:
    print(f"Warning: Groq not configured, the scene will be held and prompts use the fallback. ({e})")

FALLBACK_PROMPT = "paper lanterns drifting over a dark lake, soft haze, slow camera drift"

SYSTEM_PROMPT = (
    "You write prompts for a real-time SDXL-Turbo image stream. "
    "Turn the audio cue into ONE line of 8 to 12 short visual phrases separated by commas, "
    f"{MAX_PROMPT_WORDS} words maximum.\n"
    "Order: one to three imaginative subjects first, then their setting, surface materials, "
    "lighting colours, atmosphere (fog, particles, air movement), and camera motion last.\n"
    "The subjects are surprising things that carry the mood: creatures, animals, alien "
    "lifeforms, plants or coral, surreal objects or machines, landscape features or cosmic bodies. "
    "Mix kinds when there is more than one, and show how they relate (orbiting, growing from, "
    "chasing, carried by). Translate the music into their look and movement; never picture "
    "the music itself.\n"
    "RULES: no full sentences, no markdown, no quotes, no numbers or BPM, "
    "no genre names, no words or text to appear in the image, "
    "no people, crowds, dancers, figures, faces or bodies, "
    "no clubs, lounges, dance floors, stages, speakers, subwoofers or audio visualizers, "
    "and use glowing, luminous, crystal or neon at most once."
)

EVOLVE_PROMPT = SYSTEM_PROMPT + (
    "\nYou are revising the scene already on screen, not replacing it. Keep its main scene "
    "and most of its phrases. Change at most three phrases, preferring lighting colours, "
    "atmosphere, glowing elements and camera motion, so it fits the new audio cue. "
    "Return the whole revised line."
)

BLEND_PROMPT = SYSTEM_PROMPT + (
    "\nThe music is changing. Write a transitional line halfway between the current scene "
    "and the new audio cue: keep the setting of the current scene, and bring in the "
    "lighting, atmosphere and motion the new cue suggests, so the image can morph toward "
    "a new scene."
)

# A new scene draws its subjects from one of these, so the LLM doesn't settle on
# the same few favourites. The scene tracker picks one that wasn't used recently.
THEMES = [
    "deep ocean", "coral reef", "desert dunes", "arctic ice", "rainforest canopy",
    "volcanic land", "outer space", "ancient ruins", "clockwork machines", "mushroom forest",
    "storm sky", "mountain peaks", "swamp and marsh", "savanna at dusk", "paper and origami",
    "glass and mirrors", "mythical beasts", "giant insects up close", "birds in flight",
    "underground caves", "floating islands", "abandoned city", "autumn woodland", "tidal shore",
]


def trim_prompt(text, max_words=MAX_PROMPT_WORDS):
    """Cleans formatting and cuts to `max_words`, ending on a complete phrase."""
    text = (text.replace("\n", ", ").replace("**", "")
                .replace('"', "").replace("\u2011", "-"))
    words = text.split()
    if len(words) > max_words:
        text = " ".join(words[:max_words])
        if "," in text:
            text = text.rsplit(",", 1)[0]
    return text.strip(" ,.")


def _complete(system, user, temperature):
    """Returns {prompt, source, ms, error}; source is 'llm', 'fallback' or 'empty-reply'."""
    if not GROQ_ENABLED:
        return {"prompt": FALLBACK_PROMPT, "source": "fallback", "ms": 0,
                "error": "Groq is not configured, so the fallback prompt was used."}
    t = time.perf_counter()
    try:
        reply = groq_client.chat.completions.create(
            model=LLM_MODEL,
            messages=[{"role": "system", "content": system},
                      {"role": "user", "content": user}],
            temperature=temperature,
            max_tokens=1000,                          # gpt-oss counts its reasoning in this budget
            extra_body={"reasoning_effort": "low"},   # keeps reasoning short so the answer isn't cut off
        )
        text = (reply.choices[0].message.content or "").strip()
        ms = round((time.perf_counter() - t) * 1000)
        if not text:
            return {"prompt": FALLBACK_PROMPT, "source": "empty-reply", "ms": ms,
                    "error": "The LLM returned an empty reply, so the fallback prompt was used."}
        return {"prompt": trim_prompt(text), "source": "llm", "ms": ms, "error": None}
    except Exception as e:
        return {"prompt": FALLBACK_PROMPT, "source": "fallback",
                "ms": round((time.perf_counter() - t) * 1000), "error": f"LLM call failed: {e}"}


def generate_visual_prompt(llm_input, previous=None, theme=None, avoid=()):
    """A fresh scene. `theme` sets where the subjects come from; `previous` and
    `avoid` (recent subjects) keep the new scene from repeating earlier ones."""
    user = f"Audio cue: {llm_input}"
    if theme:
        user += f"\nTheme for this scene: {theme}. Take the subjects and setting from this theme."
    if previous:
        user += (f"\nPrevious scene: {previous}\n"
                 "Write a new scene with different subjects and a different setting.")
    if avoid:
        user += f"\nDo not use any of these subjects again: {'; '.join(avoid)}."
    return _complete(SYSTEM_PROMPT, user, 0.8)


def evolve_visual_prompt(current, llm_input):
    """The current scene, adjusted a little for the new audio."""
    return _complete(EVOLVE_PROMPT, f"Current scene: {current}\nNew audio cue: {llm_input}", 0.4)


def blend_visual_prompt(current, llm_input):
    """Halfway between the current scene and the new audio."""
    return _complete(BLEND_PROMPT, f"Current scene: {current}\nNew audio cue: {llm_input}", 0.6)

print("Prompt writers defined.")

Groq client ready (model: openai/gpt-oss-120b).
Prompt writers defined.


---
## 5. Scene tracker

Instead of a new prompt every window, a **scene** is a gradual, revisable state. The first audible window sets it. Each later window is compared with the music the scene was written for, giving a **change score** from 0 (same music) to 1 (different in every way):

| Part | Weight | Scores 1 when |
|---|---|---|
| Tempo | 0.25 | 10% tempo change (half or double time counts as the same tempo) |
| Genre | 0.25 | different parent genre (top two swapped: 0.4, same parent: 0.6) |
| Mood | 0.20 | average mood score moved by 0.25 |
| Energy | 0.20 | RMS moved 60%, or the phase jumped two steps |
| Band | 0.10 | the loudest band changed |

Parts a missing model can't provide are left out and the remaining weights scaled up. Then:

| Situation | Decision | Prompt |
|---|---|---|
| score below `SCENE_EVOLVE_AT` | **hold** | the same prompt is re-sent; no LLM call |
| `SCENE_EVOLVE_AT` to `SCENE_CHANGE_AT` | **evolve** | the LLM changes up to three phrases |
| `SCENE_CHANGE_AT` or more | **blend** | a halfway prompt; the next window confirms (**new scene**) or **reverts** |
| a silence gap (≥ `SCENE_GAP_SEC`) | **blend → new track** | a likely track change; a moderate change is enough to confirm |
| no prompt change for `SCENE_DRIFT_SEC` | **evolve** (drift) | the scene moves a little even when the music holds |
| the scene has run `SCENE_MAX_SEC` | **blend → new scene** (timed) | replaced by a scene with different subjects |
| a new file in the playlist | **blend → new track** | a known boundary: blend first, then a scene written for the new track |

If the LLM fails, the current scene is kept rather than jumping to the fallback prompt. Times are audio seconds, so a file behaves like live input.

In [15]:
import math
import random
from collections import deque

PHASE_ORDER = ["breakdown", "groove", "build", "drop"]
WEIGHTS = {"bpm": 0.25, "genre": 0.25, "mood": 0.2, "energy": 0.2, "band": 0.1}
REF_KEYS = ("bpm", "energy", "phase", "band", "primary_genre", "secondary_genre", "mood_scores")
LLM_STEPS = {"initial", "evolve", "blend", "new_scene", "new_track"}


# ---------------------------------------------------------------- change score
def _bpm_change(a, b):
    if a <= 0 or b <= 0:
        return None
    d = abs(math.log2(b / a))
    d = min(d, abs(d - 1))                # half or double time counts as the same tempo
    return min(1.0, (2 ** d - 1) / 0.10)  # a 10% tempo change scores 1


def _genre_change(a, b):
    ga, gb = a.get("primary_genre"), b.get("primary_genre")
    if not ga or not gb or "unavailable" in (ga, gb):
        return None
    if ga == gb:
        return 0.0
    if gb == a.get("secondary_genre") or ga == b.get("secondary_genre"):
        return 0.4                        # top two swapped places
    if ga.split("---")[0] == gb.split("---")[0]:
        return 0.6                        # same parent genre
    return 1.0


def _mood_change(a, b):
    ma, mb = a.get("mood_scores") or {}, b.get("mood_scores") or {}
    keys = ma.keys() & mb.keys()
    if not keys:
        return None
    return min(1.0, sum(abs(ma[k] - mb[k]) for k in keys) / len(keys) / 0.25)


def _energy_change(a, b):
    ea, eb = a["energy"], b["energy"]
    rel = abs(eb - ea) / max(ea, eb, 0.05)
    jump = abs(PHASE_ORDER.index(a["phase"]) - PHASE_ORDER.index(b["phase"]))
    return max(min(1.0, rel / 0.6), 1.0 if jump >= 2 else 0.0)


def change_score(ref, rec):
    """0 = same music as the scene was written for, 1 = different in every way."""
    parts = {
        "bpm": _bpm_change(ref["bpm"], rec["bpm"]),
        "genre": _genre_change(ref, rec),
        "mood": _mood_change(ref, rec),
        "energy": _energy_change(ref, rec),
        "band": 0.0 if ref["band"] == rec["band"] else 1.0,
    }
    parts = {k: round(v, 2) for k, v in parts.items() if v is not None}
    total = sum(WEIGHTS[k] for k in parts)
    score = sum(WEIGHTS[k] * v for k, v in parts.items()) / total
    return round(score, 3), parts


def find_gap(y, sr=SR):
    """(start, end) in seconds of the longest near-silent stretch, if it lasts at least
    SCENE_GAP_SEC. Near silent = under a tenth of the window's loud level."""
    hop = 512
    rms = librosa.feature.rms(y=y, frame_length=2048, hop_length=hop)[0]
    if not len(rms):
        return None
    loud = float(np.percentile(rms, 90))
    if loud < SCENE_SILENCE_RMS:
        return None                       # the whole window is quiet; handled as silence
    quiet = np.append(rms < 0.1 * loud, False)
    best, start = (0, 0), None
    for i, q in enumerate(quiet):
        if q and start is None:
            start = i
        elif not q and start is not None:
            if i - start > best[1] - best[0]:
                best = (start, i)
            start = None
    if (best[1] - best[0]) * hop / sr < SCENE_GAP_SEC:
        return None
    return best[0] * hop / sr, best[1] * hop / sr


# ---------------------------------------------------------------- tracker
class SceneTracker:
    def __init__(self):
        self.track = 0
        self.scene_id = 0
        self.recent_themes = deque(maxlen=6)      # kept across tracks, so a set keeps varying
        self.recent_subjects = deque(maxlen=8)
        self.prompt = None
        self.ref = None             # features the current scene was written for
        self.pending = None         # set while a change is unconfirmed
        self.decision = None
        self.last_gap_end = -1.0
        self.scene_t = None         # audio time the current scene started
        self.changed_t = None       # audio time the prompt last changed

    def describe(self):
        return {"prompt": self.prompt, "scene_id": self.scene_id, "track_no": self.track,
                "pending": self.pending is not None, "decision": self.decision}

    def start_track(self, source):
        """A known track boundary: the next file in the playlist."""
        self.track += 1
        self.ref = None
        self.last_gap_end = -1.0
        self.pending = None
        self.scene_t = self.changed_t = None     # audio time restarts with each track
        if self.prompt:
            # First window blends from the previous scene, second one replaces it
            self.pending = {"before": self.prompt, "track": True, "forced": True, "boundary": True}

    def step(self, rec, gap, use_llm):
        """Decides what this window sends, fills in the prompt fields of `rec`
        and returns the prompt (None = send nothing)."""
        if self.prompt and self.scene_t is None:
            self.scene_t = self.changed_t = rec["t_end"]
        score, parts = change_score(self.ref, rec) if self.ref else (None, {})
        gap_at = None
        if gap and rec["t_start"] + gap[0] >= self.last_gap_end:   # an overlapping window sees the same gap again
            self.last_gap_end = rec["t_start"] + gap[1]
            gap_at = round(rec["t_start"] + gap[0], 2)

        decision, note = self._decide(rec, score, gap_at)
        if decision in LLM_STEPS and not use_llm:
            decision, note = ("hold", "Prompt generation is off, so the scene is held.") if self.prompt \
                else ("wait", "Prompt generation is off and there's no scene yet.")
            self.pending = None

        llm = {"prompt": None, "source": None, "ms": 0, "error": None}
        if decision in ("initial", "new_scene", "new_track"):
            theme = random.choice([t for t in THEMES if t not in self.recent_themes])
            llm = generate_visual_prompt(rec["llm_input"],
                                         previous=self.prompt if decision != "initial" else None,
                                         theme=theme, avoid=list(self.recent_subjects))
            if llm["source"] == "llm":
                self.recent_themes.append(theme)
                self.recent_subjects.extend(p.strip() for p in llm["prompt"].split(",")[:2])
                note = f"{note} Theme: {theme}."
        elif decision == "evolve":
            llm = evolve_visual_prompt(self.prompt, rec["llm_input"])
        elif decision == "blend":
            llm = blend_visual_prompt(self.prompt, rec["llm_input"])

        if decision in LLM_STEPS and llm["source"] != "llm" and self.prompt:
            # The LLM failed: keep what's on screen rather than jump to the fallback prompt
            note = f"{llm['error']} Holding the current scene instead."
            decision, llm = "hold", {**llm, "prompt": None, "source": None}
            self.pending = None
        self._commit(decision, rec, llm["prompt"])

        source = {"hold": "held", "revert": "held",
                  "wait": "off" if not use_llm else "waiting"}.get(decision, llm["source"])
        rec.update({"prompt": self.prompt if decision != "wait" else None, "prompt_source": source,
                    "llm_ms": llm["ms"], "llm_error": llm["error"]})
        rec["scene"] = {"decision": decision, "note": note, "score": score, "parts": parts,
                        "gap_at": gap_at, **self.describe()}
        return rec["prompt"]

    def _decide(self, rec, score, gap_at):
        evolve_at, change_at = SCENE_EVOLVE_AT, SCENE_CHANGE_AT
        if rec["energy"] < SCENE_SILENCE_RMS:
            if not self.prompt:
                return "wait", "Near silence. Waiting for audio before choosing the first scene."
            if not self.pending:
                self.pending = {"before": self.prompt, "track": True, "forced": False}
            return "hold", "Near silence, so a track may be changing. Holding the scene."
        if not self.prompt:
            return "initial", "First audible window, so it sets the scene."

        p = self.pending
        if p:
            self.pending = None
            if p.get("boundary"):
                if p.get("blended"):
                    return "new_track", "New track. The scene that blended in is now replaced with one written for it."
                p["blended"] = True
                self.pending = p
                return "blend", "New track. Blending from the previous track's scene before replacing it."
            threshold = evolve_at if p["track"] else change_at
            if p["forced"] or score is None or score >= threshold:
                if p["track"]:
                    return "new_track", f"Change {score} after a gap confirms a new track (threshold {threshold})."
                return "new_scene", f"Change {score} held for a second window, so the scene is replaced."
            if self.prompt != p["before"]:
                self.prompt = p["before"]
                return "revert", f"Change fell to {score}, under {threshold}. Back to the scene from before the blend."
            return "hold", f"Change {score} after the gap is under {threshold}. Same track, scene held."

        if score is None:
            self.ref = {k: rec[k] for k in REF_KEYS}
            return "hold", "This window becomes the reference for the current prompt."
        if gap_at is not None:
            self.pending = {"before": self.prompt, "track": True, "forced": False}
            return "blend", f"Gap of near silence at {gap_at:.1f} s. Blending toward the new audio until the next window confirms."
        if score >= change_at:
            self.pending = {"before": self.prompt, "track": False, "forced": False}
            return "blend", f"Change {score} is over {change_at}. Blending; the next window confirms or reverts."
        t = rec["t_end"]
        if SCENE_MAX_SEC > 0 and t - self.scene_t >= SCENE_MAX_SEC:
            self.pending = {"before": self.prompt, "track": False, "forced": True}
            return "blend", (f"The scene has run {t - self.scene_t:.0f} s (limit {SCENE_MAX_SEC:.0f} s). "
                             "Blending; the next window replaces it with new subjects.")
        if score >= evolve_at:
            return "evolve", f"Change {score} is between {evolve_at} and {change_at}, so the scene is adjusted."
        if SCENE_DRIFT_SEC > 0 and t - self.changed_t >= SCENE_DRIFT_SEC:
            return "evolve", (f"Change {score} is small, but the prompt hasn't changed for "
                              f"{t - self.changed_t:.0f} s, so the scene drifts a little.")
        return "hold", f"Change {score} is under {evolve_at}. Same prompt re-sent."

    def _commit(self, decision, rec, new_prompt):
        if new_prompt:
            self.prompt = new_prompt
        if decision in ("initial", "new_scene", "new_track"):
            self.scene_id += 1
            self.scene_t = rec["t_end"]
        if new_prompt or decision == "revert":
            self.changed_t = rec["t_end"]
        if decision in ("initial", "new_scene", "new_track", "evolve") or (decision == "blend" and self.ref is None):
            self.ref = {k: rec[k] for k in REF_KEYS}
        self.decision = decision

print("Scene tracker defined.")

Scene tracker defined.


---
## 6. Playlist engine

Every track is cut into windows of `WINDOW_SEC` every `STEP_SEC`, and each window goes through `run_window()`:
features → sentence → **scene decision** → prompt (LLM only when the scene changes) → Control API.

Each `run_playlist()` call starts a fresh scene tracker, so the no-LLM run below doesn't affect the full run.

In [16]:
def send_to_stream(prompt):
    """Sends one prompt with the Control API client from the top of the notebook.
    Returns {ok, ms, error} and never raises."""
    t0 = time.perf_counter()
    try:
        client.send_prompt(stream_id, prompt)
        return {"ok": True, "ms": round((time.perf_counter() - t0) * 1000), "error": None}
    except NameError:
        return {"ok": False, "ms": 0, "error": "No stream: run the Mock Server Connection cells at the top."}
    except Exception as e:
        return {"ok": False, "ms": round((time.perf_counter() - t0) * 1000), "error": str(e)}


def run_window(scene, y_slice, index, t_start, t_end, use_llm=True, send=True):
    """One window through the whole pipeline. Returns its record."""
    rec = analyse_window(y_slice, sr=SR)
    gap = find_gap(y_slice)
    rec.update({"index": index, "t_start": round(t_start, 2), "t_end": round(t_end, 2)})

    # The scene decides whether this window holds, adjusts, blends or replaces the prompt
    scene.step(rec, gap, use_llm)
    rec["prompt_words"] = len(rec["prompt"].split()) if rec["prompt"] else 0

    if send and rec["prompt"]:
        rec["api"] = send_to_stream(rec["prompt"])
    else:
        rec["api"] = {"ok": None, "ms": 0, "error": None, "skipped": True}
    return rec


def mmss(s):
    m, s = divmod(int(round(s)), 60)
    return f"{m:02d}:{s:02d}"


def report(rec, track_no, total, name, update_no):
    sc, api = rec["scene"], rec["api"]
    scores = ", ".join(f"{k} {v:.2f}" for k, v in sorted(rec.get("mood_scores", {}).items(), key=lambda x: -x[1])[:3])
    print("=" * 80)
    print(f" LIVE AUDIO VIBE STREAM - UPDATE #{update_no}")
    print(f" Track {track_no:02d}/{total:02d} ({name}) | [{mmss(rec['t_start'])} -> {mmss(rec['t_end'])}]")
    print("=" * 80)
    print(f" • Tempo                   : {rec['bpm']:.1f} BPM ({rec['bpm_source']})")
    print(f" • Primary genre           : {rec['primary_genre']}")
    print(f" • Secondary genre         : {rec['secondary_genre']}")
    print(f" • Mood                    : {rec['mood']}" + (f"  ({scores})" if scores else ""))
    print(f" • Energy (RMS)            : {rec['energy']:.3f} ({rec['phase_label']})")
    print(f" • Lyric themes            : {rec['lyric_themes']}")
    print(f" • Dominant frequency band : {rec['band_label']}")
    print(" " + "-" * 78)
    print(f"  INPUT SENT TO LLM: \"{rec['llm_input']}\"")
    print(" " + "-" * 78)
    score = "" if sc["score"] is None else f"change {sc['score']:.2f}: " + ", ".join(f"{k} {v:g}" for k, v in sc["parts"].items())
    print(f"  SCENE DECISION : {sc['decision'].upper()}" + (f"  ({score})" if score else ""))
    print(f"                   {sc['note']}")
    if rec["prompt"]:
        print(f"  VISUAL PROMPT ({rec['prompt_words']}/{MAX_PROMPT_WORDS} words, {rec['prompt_source']}):")
        print(f"    {rec['prompt']}")
    if rec.get("llm_error"):
        print(f"  LLM: {rec['llm_error']}")
    if api.get("skipped"):
        print("  CONTROL API    : not sent")
    elif api["ok"]:
        print(f"  CONTROL API    : 200 OK in {api['ms']} ms -> stream {stream_id}")
    else:
        print(f"  CONTROL API    : error - {api['error']}")
    print("=" * 80 + "\n")


def run_playlist(use_llm=False, send=False):
    """Plays every track in AUDIO_PLAYLIST back to back, like a DJ set. Returns every window's record."""
    tracks = [p for p in AUDIO_PLAYLIST if os.path.exists(p)]
    for p in AUDIO_PLAYLIST:
        if p not in tracks:
            print(f" Track not found, skipped: {p}")
    print(f"\n[Playlist] {len(tracks)} track(s) | LLM {'ON' if use_llm else 'OFF'} | send {'ON' if send else 'OFF'} | "
          f"window {WINDOW_SEC:g} s every {STEP_SEC:g} s\n")
    scene, records = SceneTracker(), []
    for track_no, path in enumerate(tracks, 1):
        name = os.path.basename(path)
        y, sr = librosa.load(path, sr=SR, mono=True)
        window, step = int(WINDOW_SEC * sr), int(STEP_SEC * sr)
        scene.start_track(name)
        for i, start in enumerate(window_starts(len(y), window, step)):
            end = min(start + window, len(y))
            rec = run_window(scene, y[start:end], i, start / sr, end / sr, use_llm=use_llm, send=send)
            rec["track"] = name
            records.append(rec)
            report(rec, track_no, len(tracks), name, i + 1)
    return records

print("Engine defined.")

Engine defined.


### Run the playlist (no LLM, nothing sent)
Shows the features and input sentence for every window. Without the LLM the scene can't change, so decisions show **wait** or **hold**.

In [17]:
records_no_llm = run_playlist(use_llm=False, send=False)


[Playlist] 3 track(s) | LLM OFF | send OFF | window 10 s every 5 s

 LIVE AUDIO VIBE STREAM - UPDATE #1
 Track 01/03 (036-1896735-Revolution Void-Effects of Elevation _Citizen Prime Remix_.mp3) | [00:00 -> 00:10]
 • Tempo                   : 133.3 BPM (Madmom neural)
 • Primary genre           : Electronic---Progressive Breaks
 • Secondary genre         : Electronic---House
 • Mood                    : party, relaxed  (party 0.83, relaxed 0.55, happy 0.25)
 • Energy (RMS)            : 0.118 (Low Energy / Breakdown Phase)
 • Lyric themes            : instrumental
 • Dominant frequency band : Low-End / Sub-Bass Focus (20 Hz - 120 Hz)
 ------------------------------------------------------------------------------
  INPUT SENT TO LLM: "A party, relaxed mood, Low Energy / Breakdown Phase 133.3 BPM Progressive Breaks atmosphere, featuring Low-End / Sub-Bass Focus with instrumental themes."
 ------------------------------------------------------------------------------
  SCENE DECISION : WAI

---
## 7. Full pipeline: LLM and Control API
### 7.1 Check the LLM

In [18]:
print(f"Groq client ready (model: {LLM_MODEL})." if GROQ_ENABLED else
      "Warning: Groq not configured. The scene can't change and prompts use the fallback.")

Groq client ready (model: openai/gpt-oss-120b).


### 7.2 Full pipeline
Audio → sentence → scene decision → LLM → prompt → stream, for every window of every track. Uses the `client` and `stream_id` created in the Mock Server Connection cells at the top. Records are saved to `/content/results/pipeline-<time>.csv`.

In [19]:
import csv
from datetime import datetime

records = run_playlist(use_llm=True, send=True)

os.makedirs("/content/results", exist_ok=True)
FIELDS = ["track", "index", "t_start", "t_end", "bpm", "bpm_source", "energy", "phase", "band", "primary_genre",
          "secondary_genre", "mood", "lyric_themes", "llm_input", "prompt", "prompt_source", "analysis_ms", "llm_ms"]
out = f"/content/results/pipeline-{datetime.now():%Y%m%d-%H%M%S}.csv"
with open(out, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(FIELDS + ["decision", "change_score", "api_ok"])
    for r in records:
        w.writerow([r.get(k) for k in FIELDS] + [r["scene"]["decision"], r["scene"]["score"], r["api"].get("ok")])

decisions = Counter(r["scene"]["decision"] for r in records)
sent = [r["api"] for r in records if not r["api"].get("skipped")]
print(f"{len(records)} windows | decisions {dict(decisions)} | sent ok {sum(bool(a['ok']) for a in sent)}/{len(sent)}")
print("Saved", out)


[Playlist] 3 track(s) | LLM ON | send ON | window 10 s every 5 s

 LIVE AUDIO VIBE STREAM - UPDATE #1
 Track 01/03 (036-1896735-Revolution Void-Effects of Elevation _Citizen Prime Remix_.mp3) | [00:00 -> 00:10]
 • Tempo                   : 133.3 BPM (Madmom neural)
 • Primary genre           : Electronic---Progressive Breaks
 • Secondary genre         : Electronic---House
 • Mood                    : party, relaxed  (party 0.83, relaxed 0.55, happy 0.25)
 • Energy (RMS)            : 0.118 (Low Energy / Breakdown Phase)
 • Lyric themes            : instrumental
 • Dominant frequency band : Low-End / Sub-Bass Focus (20 Hz - 120 Hz)
 ------------------------------------------------------------------------------
  INPUT SENT TO LLM: "A party, relaxed mood, Low Energy / Breakdown Phase 133.3 BPM Progressive Breaks atmosphere, featuring Low-End / Sub-Bass Focus with instrumental themes."
 ------------------------------------------------------------------------------
  SCENE DECISION : INITI

In [20]:
print(open('server.log').read())

Traceback (most recent call last):
  File "/content/mock-control-api.py", line 311, in <module>
    main()
    ~~~~^^
  File "/content/mock-control-api.py", line 295,  "GET /api/streams HTTP/1.1" 401 -
  login test@example.com -> token
  "POST /api/auth HTTP/1.1" 200 -
  created str_b22275f29e549e6a (ready in 4s)
  "POST /api/streams HTTP/1.1" 201 -

  File "/usr/lib/python3.13/http/server.py", line 140, in server_bind
    socketserver.TCPServer.server_bind(self)
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^
  File "/usr/lib/python3.13/socketserver.py", line 478, in server_bind
    self.socket.bind(self.server_address)
    ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^
OSError: [Errno 98] Address already in use



---
## 8. Evaluation: baseline vs new pipeline

Both pipelines are scored on **the same windows** from section 7.2 (or the no-LLM run if 7.2 was skipped), so no extra audio analysis is needed. Nothing is sent to the stream.

| | Baseline | New pipeline |
|---|---|---|
| Features | Librosa only: tempo, energy phase, loudest band | Madmom tempo, Essentia genre and mood, Whisper lyric themes |
| Genre | always the most common label in `labels.csv` (majority class) | Essentia Discogs-400 |
| Prompt | a fixed template filled in from phase and band, no LLM | written by the LLM |
| Scenes | none; the prompt is rebuilt every window | scene tracking: hold, evolve, blend, replace |

**Ground truth is optional**, in `LABELS_CSV` (file names as in `AUDIO_PLAYLIST`):

```
file,genre,bpm,boundaries
001-2286013-Play House-Marijuana.mp3,house,124,
mix.mp3,,,95.5;201
```

`genre` and `bpm` score feature accuracy. `boundaries` (seconds, `;`-separated) mark where one song ends and the next begins in a file of joined songs, and score whether scene changes happen where the music changes. Without labels, the accuracy rows show `n/a` and the stability, scene and timing rows still compare the two.

Set `RUN_JUDGE = True` in Configuration to have the LLM score each prompt's fit to the music (1–5). The judge sees the new pipeline's description of the music, which the new prompt was written from, so treat that score as a guide, not proof.

In [21]:
import csv
import re
from datetime import datetime
import statistics
import pandas as pd
from IPython.display import Markdown, display

# --- Baseline: template prompt from phase and band ------------------------------
PHASE_PHRASES = {
    "breakdown": "dim empty space, soft blue haze, slow drift",
    "groove": "night city street, warm amber light, steady pan",
    "build": "tunnel of rising light, magenta glow, camera pushing forward",
    "drop": "neon arena, white strobes, fast camera shake",
}
BAND_PHRASES = {"sub": "deep pulsing shapes", "mid": "glowing synth ribbons", "high": "glittering sparks"}
SCENE_STARTS = {"new_scene", "new_track"}

def baseline_prompt(rec):
    return f"{PHASE_PHRASES[rec['phase']]}, {BAND_PHRASES[rec['band']]}"

# --- Scoring helpers ----------------------------------------------------------------
def norm(s):
    return re.sub(r"[^a-z0-9]", "", (s or "").lower())

def genre_match(label, predicted):
    """'metal' matches 'Rock---Heavy Metal'; 'hiphop' matches 'Hip Hop---Trap'."""
    return bool(label) and norm(label) in norm(predicted)

def tempo_hits(est, truth):
    """(acc1, acc2): within 4% of the true tempo; acc2 also accepts x2, x3, /2, /3."""
    if not est or not truth:
        return None, None
    acc1 = abs(est - truth) <= 0.04 * truth
    acc2 = any(abs(est - truth * f) <= 0.04 * truth * f for f in (1, 2, 3, 1 / 2, 1 / 3))
    return acc1, acc2

JUDGE_PROMPT = ("You rate prompts for a real-time image stream that visualises music. "
                "Given a description of the music and a visual prompt, rate how well the "
                "prompt fits the music on a scale of 1 (no fit) to 5 (excellent fit), "
                "considering mood, energy, tempo and genre. Reply with the digit only.")

def judge(music, visual):
    """1-5 from the LLM, or None if it isn't available or answers oddly."""
    if not GROQ_ENABLED or not visual:
        return None
    try:
        reply = groq_client.chat.completions.create(
            model=LLM_MODEL,
            messages=[{"role": "system", "content": JUDGE_PROMPT},
                      {"role": "user", "content": f"Music: {music}\nPrompt: {visual}"}],
            temperature=0, max_tokens=400, extra_body={"reasoning_effort": "low"})
        m = re.search(r"[1-5]", reply.choices[0].message.content or "")
        return int(m.group()) if m else None
    except Exception as e:
        print(f"    judge failed: {e}")
        return None

def read_labels(path):
    if not path or not os.path.exists(path):
        return {}
    rows = {}
    with open(path, newline="", encoding="utf-8") as f:
        for r in csv.DictReader(f):
            bounds = [float(b) for b in (r.get("boundaries") or "").split(";") if b.strip()]
            rows[r["file"].strip()] = {"genre": (r.get("genre") or "").strip(),
                                       "bpm": float(r["bpm"]) if (r.get("bpm") or "").strip() else None,
                                       "boundaries": bounds}
    return rows

def boundary_scores(rows, boundaries, starts_key):
    """(boundaries caught, total, scene changes not near a boundary). A boundary is caught if a
    scene starts in a window ending within one window plus one step after it (blend, then confirm)."""
    starts = [r["t_end"] for r in rows if r[starts_key]]
    near = lambda t, b: b <= t <= b + WINDOW_SEC + STEP_SEC
    caught = sum(any(near(t, b) for t in starts) for b in boundaries)
    spurious = sum(not any(near(t, b) for b in boundaries) for t in starts)
    return caught, len(boundaries), spurious

def mean(xs):
    xs = [x for x in xs if x is not None]
    return statistics.mean(xs) if xs else None

def pct(xs):
    xs = [x for x in xs if x is not None]
    return 100 * sum(xs) / len(xs) if xs else None

def fmt(v, unit="", digits=1):
    return "n/a" if v is None else f"{v:.{digits}f}{unit}"

print("Comparison helpers defined.")

Comparison helpers defined.


In [25]:
# --- Pick the records to score --------------------------------------------------------
if "records" in globals() and records:
    eval_records, llm_note = records, "full pipeline run (7.2)"
else:
    eval_records, llm_note = records_no_llm, "no-LLM run: the new pipeline holds one scene, so its scene rows mean little"
labels = read_labels(LABELS_CSV)
print(f"Scoring {len(eval_records)} windows from the {llm_note}. Labels: {len(labels) or 'none'}")

STAGES = ["librosa", "bands", "madmom", "essentia", "whisper"]   # analysis steps timed in analyse_window()
BASE_STAGES = ("librosa", "bands")                                 # all the baseline needs
TEMPO_JUMP = 0.25                                                  # a 25%+ tempo change between windows counts as a jump

def short(name, n=28):
    name = os.path.splitext(name)[0]
    return name if len(name) <= n else name[:n - 1] + "…"

def mmss_f(sec):
    m, s = divmod(int(round(sec)), 60)
    return f"{m}:{s:02d}"

# --- One row per window, both pipelines side by side ----------------------------------
rows, prev = [], {}            # prev[track] = (base_prompt, new_prompt, base_bpm, new_bpm, scene note)
for rec in eval_records:
    lab = labels.get(rec["track"], {})
    base, new = baseline_prompt(rec), rec["prompt"]
    pb, pn, bb, nb_, pnote = prev.get(rec["track"], (None, None, None, None, ""))
    t, sc, api = rec["timings"], rec["scene"], rec.get("api", {})
    decision = sc["decision"]

    # Why a scene started: the playlist moved on, the music changed, or the SCENE_MAX_SEC timer ran out
    scene_cause = {"new_track": "track", "initial": "first"}.get(decision)
    if decision == "new_scene":
        scene_cause = "timer" if pnote.startswith("The scene has run") else "music"
    evolve_cause = ("drift" if "hasn't changed for" in sc["note"] else "music") if decision == "evolve" else None

    row = {
        "track": rec["track"], "window": rec["index"], "t_end": rec["t_end"],
        "energy": rec["energy"], "phase": rec["phase"], "band": rec["band"],
        "true_bpm": lab.get("bpm"), "base_bpm": rec["librosa_bpm"], "new_bpm": rec["bpm"],
        "true_genre": lab.get("genre", ""), "new_genre": rec["primary_genre"], "new_mood": rec["mood"],
        "music_description": rec["llm_input"],
        "base_prompt": base, "new_prompt": new or "", "new_decision": decision,
        "new_scene_cause": scene_cause, "new_evolve_cause": evolve_cause,
        "new_prompt_source": rec["prompt_source"],
        "base_changed": pb is not None and base != pb,
        "new_changed": pn is not None and new is not None and new != pn,
        "base_scene_start": pb is not None and base != pb,
        "new_scene_start": decision in SCENE_STARTS,
        "base_tempo_jump": bool(bb) and bool(rec["librosa_bpm"]) and abs(rec["librosa_bpm"] / bb - 1) >= TEMPO_JUMP,
        "new_tempo_jump": bool(nb_) and bool(rec["bpm"]) and abs(rec["bpm"] / nb_ - 1) >= TEMPO_JUMP,
        **{f"{k}_ms": t.get(k, 0) for k in STAGES},
        "llm_ms": rec.get("llm_ms", 0) or 0,
        "send_ms": api.get("ms", 0) or 0,
        "api_ok": None if api.get("skipped") else bool(api.get("ok")),
        "api_error": api.get("error"),
        "llm_error": rec.get("llm_error"),
        "base_ms": sum(t.get(k, 0) for k in BASE_STAGES),
        "new_ms": rec["analysis_ms"] + (rec.get("llm_ms", 0) or 0),
        "base_judge": None, "new_judge": None,
    }
    if RUN_JUDGE:
        row["base_judge"] = judge(rec["llm_input"], base)
        row["new_judge"] = judge(rec["llm_input"], new)
    rows.append(row)
    prev[rec["track"]] = (base, new if new is not None else pn, rec["librosa_bpm"], rec["bpm"], sc["note"])

cmp_df = pd.DataFrame(rows)
tracks = list(dict.fromkeys(r["track"] for r in rows))
audio_sec = sum(cmp_df[cmp_df.track == tr].t_end.max() for tr in tracks)
minutes = audio_sec / 60
step_ms = STEP_SEC * 1000
n = len(rows)
llm_prompts = sum(r["new_prompt_source"] == "llm" for r in rows)
wall_sec = globals().get("run_wall_sec")          # set by the 7.2 cell

# =====================================================================================
# 1. Run time
# =====================================================================================
base_total, new_total = cmp_df.base_ms.sum() / 1000, cmp_df.new_ms.sum() / 1000
send_total = cmp_df.send_ms.sum() / 1000
base_ontime = 100 * (cmp_df.base_ms <= step_ms).mean()
new_ontime = 100 * (cmp_df.new_ms <= step_ms).mean()

run_rows = [
    ("Audio processed", mmss_f(audio_sec), mmss_f(audio_sec), f"{len(tracks)} tracks, {n} windows"),
    ("Total processing time", mmss_f(base_total) if base_total >= 60 else f"{base_total:.1f} s", mmss_f(new_total),
     "all windows added up (analysis + LLM)"),
    ("Time per window, typical", f"{cmp_df.base_ms.median():.0f} ms", f"{cmp_df.new_ms.median():.0f} ms", "median"),
    ("Time per window, slowest", f"{cmp_df.base_ms.max():.0f} ms", f"{cmp_df.new_ms.max():.0f} ms", ""),
    (f"Windows finished within {STEP_SEC:g} s", f"{base_ontime:.0f}%", f"{new_ontime:.0f}%",
     f"a new window arrives every {STEP_SEC:g} s; under 100% means it falls behind live audio"),
]
if wall_sec:
    run_rows.append(("Wall-clock time of the whole 7.2 run", "", mmss_f(wall_sec),
                     f"includes loading audio, printing and sending ({send_total:.1f} s sending)"))

stage_rows = []
for k in STAGES + ["llm", "send"]:
    col = cmp_df[f"{k}_ms"]
    if col.sum() == 0:
        continue
    used = col[col > 0]
    label = {"llm": "LLM (prompt writing)", "send": "Control API send", "bands": "frequency bands"}.get(k, k.capitalize())
    stage_rows.append((label, f"{used.median():.0f} ms", f"{col.max():.0f} ms",
                       f"{col.sum() / 1000:.1f} s", f"{100 * col.sum() / max(cmp_df.new_ms.sum() + cmp_df.send_ms.sum(), 1):.0f}%",
                       "yes" if k in BASE_STAGES else "no"))

track_rows = []
for tr in tracks:
    g = cmp_df[cmp_df.track == tr]
    track_rows.append((short(tr), mmss_f(g.t_end.max()), len(g), f"{g.base_ms.sum() / 1000:.1f} s",
                       mmss_f(g.new_ms.sum() / 1000), f"{g.new_ms.max():.0f} ms",
                       f"{100 * (g.new_ms <= step_ms).mean():.0f}%"))

# =====================================================================================
# 2. What the viewer would see (no labels needed)
# =====================================================================================
causes = cmp_df.new_scene_cause.value_counts()
n_music, n_timer, n_track = causes.get("music", 0), causes.get("timer", 0), causes.get("track", 0)
held_pct = 100 * (cmp_df.new_prompt_source == "held").mean()
sent = cmp_df[cmp_df.api_ok.notna()]
api_errors = sent[sent.api_ok == False].api_error.dropna()
llm_errors = cmp_df.llm_error.dropna()

def per_min(x):
    return f"{x / minutes:.2f}" if minutes else "n/a"

behaviour_rows = [
    ("Different prompts used", cmp_df.base_prompt.nunique(), cmp_df.new_prompt.replace("", None).nunique(),
     "how many distinct pictures were asked for"),
    ("Prompt changes per minute", per_min(cmp_df.base_changed.sum()), per_min(cmp_df.new_changed.sum()),
     "any change to the text sent, big or small"),
    ("Scene changes per minute", per_min(cmp_df.base_scene_start.sum()), per_min(cmp_df.new_scene_start.sum()),
     "a completely new picture"),
    ("… because the track changed", "–", n_track, "baseline can't tell"),
    ("… because the music changed", cmp_df.base_scene_start.sum(), n_music,
     "baseline: phase or band flipped"),
    ("… because of the timer", "–", n_timer, f"scene older than SCENE_MAX_SEC ({SCENE_MAX_SEC:g} s)"),
    ("Windows that kept the same prompt", f"{100 * (~cmp_df.base_changed).mean():.0f}%", f"{held_pct:.0f}%",
     "higher = calmer screen"),
    ("Tempo jumps", int(cmp_df.base_tempo_jump.sum()), int(cmp_df.new_tempo_jump.sum()),
     f"tempo moved {TEMPO_JUMP:.0%}+ from the last window; usually a half/double-time error, lower is better"),
]
if len(sent):
    behaviour_rows.append(("Prompts delivered to the stream", "not sent", f"{int(sent.api_ok.sum())}/{len(sent)}",
                           api_errors.iloc[0] if len(api_errors) else ""))
if len(llm_errors):
    behaviour_rows.append(("LLM failures (scene held instead)", "–", len(llm_errors), llm_errors.iloc[0][:80]))

# =====================================================================================
# 3. Accuracy (only with labels.csv and/or RUN_JUDGE)
# =====================================================================================
audible = [r for r in rows if r["energy"] >= SCENE_SILENCE_RMS]
s = {}
tempo = [r for r in audible if r["true_bpm"]]
for side in ("base", "new"):
    hits = [tempo_hits(r[f"{side}_bpm"], r["true_bpm"]) for r in tempo]
    s[f"{side}_tempo_acc1"] = pct(h[0] for h in hits)
    s[f"{side}_tempo_acc2"] = pct(h[1] for h in hits)
genre = [r for r in audible if r["true_genre"]]
track_genres = [lab["genre"] for lab in labels.values() if lab["genre"]]
majority = Counter(norm(g) for g in track_genres).most_common(1)[0][0] if track_genres else None
s["base_genre_acc"] = pct(norm(r["true_genre"]) == majority for r in genre) if majority else None
s["new_genre_acc"] = pct(genre_match(r["true_genre"], r["new_genre"]) for r in genre) \
    if any(r["new_genre"] != "unavailable" for r in genre) else None
n_bounds = 0
for side in ("base", "new"):
    caught = total = spurious = 0
    for track, lab in labels.items():
        if lab["boundaries"]:
            c, tt, sp = boundary_scores([r for r in rows if r["track"] == track], lab["boundaries"], f"{side}_scene_start")
            caught, total, spurious = caught + c, total + tt, spurious + sp
    s[f"{side}_boundary_recall"] = 100 * caught / total if total else None
    s[f"{side}_boundary_spurious"] = spurious if total else None
    n_bounds = total
    s[f"{side}_judge"] = mean(r[f"{side}_judge"] for r in rows)

acc_rows = [
    ("Tempo correct (within 4%)", s["base_tempo_acc1"], s["new_tempo_acc1"], "%", f"{len(tempo)} labelled windows"),
    ("Tempo correct, half/double allowed", s["base_tempo_acc2"], s["new_tempo_acc2"], "%", ""),
    ("Genre correct", s["base_genre_acc"], s["new_genre_acc"], "%",
     f"baseline always guesses '{majority}'" if majority else ""),
    ("Song changes caught", s["base_boundary_recall"], s["new_boundary_recall"], "%", f"{n_bounds} labelled boundaries"),
    ("Scene changes away from a song change", s["base_boundary_spurious"], s["new_boundary_spurious"], "", "lower is better"),
    ("Prompt fit, LLM judge (1-5)", s["base_judge"], s["new_judge"], "", "the judge favours the new pipeline"),
]
acc_rows = [(a, fmt(b, u, 0 if u == "" and "changes" in a else 1), fmt(c, u, 0 if u == "" and "changes" in a else 1), d)
            for a, b, c, u, d in acc_rows if b is not None or c is not None]

# =====================================================================================
# Plain-language summary + tables
# =====================================================================================
keeps_up = ("keeps up with live audio" if new_ontime == 100 else
            "just about keeps up with live audio" if new_ontime >= 90 else "can't keep up with live audio")
summary = [
    f"- **Speed:** the new pipeline needed **{mmss_f(new_total)}** to process **{mmss_f(audio_sec)}** of audio "
    f"(baseline: {base_total:.1f} s). {new_ontime:.0f}% of its windows finished inside the {STEP_SEC:g} s step, "
    f"so it {keeps_up}. Slowest step: **{max(stage_rows, key=lambda r: float(r[3][:-2]))[0]}**." if stage_rows else "",
    f"- **Variety:** it asked for **{cmp_df.new_prompt.replace('', None).nunique()}** different pictures; "
    f"the baseline only ever has {cmp_df.base_prompt.nunique()}.",
    f"- **Scene changes:** {int(cmp_df.new_scene_start.sum())} in total: {n_track} at track changes, "
    f"{n_music} because the music changed, {n_timer} because a scene hit the {SCENE_MAX_SEC:g} s limit.",
    f"- **Tempo stability:** the tempo jumped {int(cmp_df.new_tempo_jump.sum())} times "
    f"(baseline {int(cmp_df.base_tempo_jump.sum())}). Jumps usually mean the detector halved or doubled the beat.",
]
if len(sent) and sent.api_ok.sum() < len(sent):
    ok = int(sent.api_ok.sum())
    summary.append(f"- ⚠️ **{'None' if ok == 0 else f'Only {ok}'} of the {len(sent)} prompts reached the stream.** "
                   f"Error: `{api_errors.iloc[0] if len(api_errors) else 'unknown'}`")
if not acc_rows:
    summary.append("- **Accuracy isn't measured in this run** (no labels, judge off). See part 3.")

def table(header, body):
    return "\n".join(["| " + " | ".join(header) + " |", "|" + "---|" * len(header),
                      *["| " + " | ".join(str(x) for x in r) + " |" for r in body]])

md = [
    "## Baseline vs new pipeline", "",
    f"{len(tracks)} tracks, {n} windows of {WINDOW_SEC:g} s, one every {STEP_SEC:g} s. "
    f"LLM: {LLM_MODEL if GROQ_ENABLED else 'off'} ({llm_prompts} prompts written by it).", "",
    "**In short**", "", *[x for x in summary if x], "",
    "### 1. Run time", "",
    table(["", "Baseline", "New pipeline", "Notes"], run_rows), "",
    "**Where the time goes (new pipeline, per window)**", "",
    table(["Step", "Typical", "Slowest", "Total", "Share", "Baseline uses it"], stage_rows), "",
    "**Per track**", "",
    table(["Track", "Length", "Windows", "Baseline", "New pipeline", "Slowest window", f"Within {STEP_SEC:g} s"], track_rows), "",
    "### 2. What changes on screen", "",
    table(["", "Baseline", "New pipeline", "What it means"], behaviour_rows), "",
    "### 3. Accuracy", "",
]
if acc_rows:
    md.append(table(["", "Baseline", "New pipeline", "Notes"], acc_rows))
else:
    md += [f"Skipped: there's no `{LABELS_CSV}` and `RUN_JUDGE` is off, so there's nothing to check the answers against. "
           "To turn this on, upload a labels file like the one below (one line per track, `boundaries` can stay empty) "
           "and/or set `RUN_JUDGE = True` in Configuration, then re-run this cell.", "",
           "```", "file,genre,bpm,boundaries", *[f"{os.path.basename(tr)},,," for tr in tracks], "```"]
display(Markdown("\n".join(md)))

os.makedirs("/content/results", exist_ok=True)
cmp_out = f"/content/results/compare-{datetime.now():%Y%m%d-%H%M%S}.csv"
cmp_df.to_csv(cmp_out, index=False)
print("Per-window comparison saved to", cmp_out)


Scoring 180 windows from the full pipeline run (7.2). Labels: none


## Baseline vs new pipeline

3 tracks, 180 windows of 10 s, one every 5 s. LLM: openai/gpt-oss-120b (93 prompts written by it).

**In short**

- **Speed:** the new pipeline needed **14:35** to process **15:15** of audio (baseline: 6.5 s). 61% of its windows finished inside the 5 s step, so it can't keep up with live audio. Slowest step: **Essentia**.
- **Variety:** it asked for **93** different pictures; the baseline only ever has 4.
- **Scene changes:** 20 in total: 2 at track changes, 10 because the music changed, 8 because a scene hit the 60 s limit.
- **Tempo stability:** the tempo jumped 17 times (baseline 19). Jumps usually mean the detector halved or doubled the beat.
- ⚠️ **None of the 180 prompts reached the stream.** Error: `HTTPConnectionPool(host='127.0.0.1', port=8787): Max retries exceeded with url: /api/streams (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7b24e1b0c910>: Failed to establish a new connection: [Errno 111] Connection refused'))`
- **Accuracy isn't measured in this run** (no labels, judge off). See part 3.

### 1. Run time

|  | Baseline | New pipeline | Notes |
|---|---|---|---|
| Audio processed | 15:15 | 15:15 | 3 tracks, 180 windows |
| Total processing time | 6.5 s | 14:35 | all windows added up (analysis + LLM) |
| Time per window, typical | 34 ms | 4780 ms | median |
| Time per window, slowest | 74 ms | 7597 ms |  |
| Windows finished within 5 s | 100% | 61% | a new window arrives every 5 s; under 100% means it falls behind live audio |

**Where the time goes (new pipeline, per window)**

| Step | Typical | Slowest | Total | Share | Baseline uses it |
|---|---|---|---|---|---|
| Librosa | 24 ms | 61 ms | 4.6 s | 1% | yes |
| frequency bands | 10 ms | 18 ms | 1.9 s | 0% | yes |
| Madmom | 1984 ms | 2242 ms | 358.1 s | 41% | no |
| Essentia | 2339 ms | 2559 ms | 415.8 s | 47% | no |
| Whisper | 49 ms | 1464 ms | 12.2 s | 1% | no |
| LLM (prompt writing) | 776 ms | 2625 ms | 81.9 s | 9% | no |
| Control API send | 1 ms | 4 ms | 0.2 s | 0% | no |

**Per track**

| Track | Length | Windows | Baseline | New pipeline | Slowest window | Within 5 s |
|---|---|---|---|---|---|---|
| 036-1896735-Revolution Void… | 7:20 | 87 | 3.2 s | 7:19 | 7597 ms | 49% |
| 046-1875274-Axined _amp_ Mr… | 4:45 | 56 | 2.0 s | 4:28 | 6082 ms | 61% |
| 003-2235411-Jackson Frost-F… | 3:10 | 37 | 1.3 s | 2:48 | 5650 ms | 86% |

### 2. What changes on screen

|  | Baseline | New pipeline | What it means |
|---|---|---|---|
| Different prompts used | 4 | 93 | how many distinct pictures were asked for |
| Prompt changes per minute | 1.25 | 5.97 | any change to the text sent, big or small |
| Scene changes per minute | 1.25 | 1.31 | a completely new picture |
| … because the track changed | – | 2 | baseline can't tell |
| … because the music changed | 19 | 10 | baseline: phase or band flipped |
| … because of the timer | – | 8 | scene older than SCENE_MAX_SEC (60 s) |
| Windows that kept the same prompt | 89% | 48% | higher = calmer screen |
| Tempo jumps | 19 | 17 | tempo moved 25%+ from the last window; usually a half/double-time error, lower is better |
| Prompts delivered to the stream | not sent | 0/180 | HTTPConnectionPool(host='127.0.0.1', port=8787): Max retries exceeded with url: /api/streams (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7b24e1b0c910>: Failed to establish a new connection: [Errno 111] Connection refused')) |

### 3. Accuracy

Skipped: there's no `/content/labels.csv` and `RUN_JUDGE` is off, so there's nothing to check the answers against. To turn this on, upload a labels file like the one below (one line per track, `boundaries` can stay empty) and/or set `RUN_JUDGE = True` in Configuration, then re-run this cell.

```
file,genre,bpm,boundaries
036-1896735-Revolution Void-Effects of Elevation _Citizen Prime Remix_.mp3,,,
046-1875274-Axined _amp_ Mr. Moohman-How Do I Get My Grass Really Green.mp3,,,
003-2235411-Jackson Frost-Fashionable Progressive House.mp3,,,
```

Per-window comparison saved to /content/results/compare-20261020-105337.csv


### Prompts side by side
The baseline rebuilds its template every window; the new pipeline's decision column shows when and why its prompt changed.

In [23]:
pd.set_option("display.max_colwidth", 90)
display(cmp_df[["track", "t_end", "phase", "band", "new_decision", "base_prompt", "new_prompt"]])

,track,t_end,phase,band,new_decision,base_prompt,new_prompt
0,036-1896735-Revolution Void-Effects of Elevation _Citizen Prime Remix_.mp3,10.0,breakdown,sub,initial,"dim empty space, soft blue haze, slow drift, deep pulsing shapes","floating glass shards, mirrored jellyfish, luminescent crystal insects, endless hall o..."
1,036-1896735-Revolution Void-Effects of Elevation _Citizen Prime Remix_.mp3,15.0,breakdown,sub,hold,"dim empty space, soft blue haze, slow drift, deep pulsing shapes","floating glass shards, mirrored jellyfish, luminescent crystal insects, endless hall o..."
2,036-1896735-Revolution Void-Effects of Elevation _Citizen Prime Remix_.mp3,20.0,breakdown,sub,hold,"dim empty space, soft blue haze, slow drift, deep pulsing shapes","floating glass shards, mirrored jellyfish, luminescent crystal insects, endless hall o..."
3,036-1896735-Revolution Void-Effects of Elevation _Citizen Prime Remix_.mp3,25.0,breakdown,sub,evolve,"dim empty space, soft blue haze, slow drift, deep pulsing shapes","floating glass shards, mirrored jellyfish, luminescent crystal insects, endless hall o..."
4,036-1896735-Revolution Void-Effects of Elevation _Citizen Prime Remix_.mp3,30.0,breakdown,sub,evolve,"dim empty space, soft blue haze, slow drift, deep pulsing shapes","floating glass shards, mirrored jellyfish, luminescent crystal insects, endless hall o..."
...,...,...,...,...,...,...,...
175,003-2235411-Jackson Frost-Fashionable Progressive House.mp3,170.0,groove,sub,hold,"night city street, warm amber light, steady pan, deep pulsing shapes","caramel-coated leaf dragons, amber moth swarms, copper acorn orbs, fallen maple carpet..."
176,003-2235411-Jackson Frost-Fashionable Progressive House.mp3,175.0,groove,sub,evolve,"night city street, warm amber light, steady pan, deep pulsing shapes","caramel-coated leaf dragons, amber moth swarms, copper acorn orbs, fallen maple carpet..."
177,003-2235411-Jackson Frost-Fashionable Progressive House.mp3,180.0,groove,sub,hold,"night city street, warm amber light, steady pan, deep pulsing shapes","caramel-coated leaf dragons, amber moth swarms, copper acorn orbs, fallen maple carpet..."
178,003-2235411-Jackson Frost-Fashionable Progressive House.mp3,185.0,groove,sub,evolve,"night city street, warm amber light, steady pan, deep pulsing shapes","caramel-coated leaf dragons, amber moth swarms, copper acorn orbs, fallen maple carpet..."
